# GSE193531 — Step 3: Cell-Type Annotation

Scores every cell against 8 canonical marker sets, aggregates scores to the
Leiden-cluster level, assigns each cluster to its highest-scoring cell type
(flagging ambiguous clusters), cross-checks that call against each cluster's
own unbiased marker genes (overriding only where the raw-score call is
directly contradicted), and maps the result back onto every cell.

This cohort (GSE193531) is CD138-sorted plasma cells, so `CD4_T`/`CD8_T`/
`NK_cell` fractions are expected to come out at or near zero for nearly every
cluster — the same 8-marker-set template is applied across every cohort in
this project for schema consistency, so that is the correct, informative
result, not a bug.

**Input:** `data/processed/GSE193531/GSE193531_normalized.h5ad` (from notebook 02)

**Output:**
- `data/processed/GSE193531/GSE193531_annotated.h5ad`
- `results/GSE193531/cell_annotation/GSE193531_cluster_annotation_summary.csv`

This step is fully mechanical — no run-specific hardcoded constants.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc

sc.settings.verbosity = 1

#### Directory layout and required files

In [ ]:
PROJECT = Path("../")
COHORT = "GSE193531"

PROCESSED_DIR = PROJECT / "data" / "processed" / COHORT
RESULTS_DIR = PROJECT / "results" / COHORT

(RESULTS_DIR / "cell_annotation").mkdir(parents=True, exist_ok=True)

INPUT_H5AD = PROCESSED_DIR / f"{COHORT}_normalized.h5ad"
if not INPUT_H5AD.exists():
    raise FileNotFoundError(INPUT_H5AD)

OUTPUT_H5AD = PROCESSED_DIR / f"{COHORT}_annotated.h5ad"

## Load Normalized AnnData

In [ ]:
adata = sc.read_h5ad(INPUT_H5AD)
print(adata)
print()
print(adata.obs["cell_type_candidate"].value_counts())

## Canonical Marker Gene Sets

The same 8 cell types are used for every cohort in this project, so
patient-level features stay comparable across cohorts even when a cohort's
sort strategy means some types are barely present. Markers are filtered to
genes actually present in this cohort's `var_names`.

In [ ]:
marker_sets_raw = {
    "Plasma_cell": ["MZB1", "SDC1", "PRDM1", "XBP1", "JCHAIN", "CD38", "TNFRSF17", "DERL3", "SLAMF7", "FCRL5"],
    "B_cell": ["MS4A1", "CD79A", "CD79B", "CD19", "CD74", "HLA-DRA", "CD37"],
    "CD4_T": ["CD3D", "CD3E", "CD4", "IL7R", "CD40LG"],
    "CD8_T": ["CD3D", "CD3E", "CD8A", "CD8B", "GZMK"],
    "NK_cell": ["GNLY", "NKG7", "KLRD1", "KLRF1", "NCAM1"],
    "Myeloid": ["LYZ", "CD14", "S100A8", "S100A9", "CST3", "FCN1"],
    "pDC": ["IRF8", "TCF4", "IRF7", "PLAC8", "LILRA4"],
    "Erythroid": ["HBA1", "HBA2", "HBB", "GYPA", "ALAS2"],
}

MIN_MARKERS_PER_TYPE = 2

marker_sets = {}
for cell_type, genes in marker_sets_raw.items():
    present = [g for g in genes if g in adata.var_names]
    missing = [g for g in genes if g not in adata.var_names]
    if missing:
        print(f"{cell_type}: missing {missing}")
    if len(present) < MIN_MARKERS_PER_TYPE:
        print(f"{cell_type}: only {len(present)} marker(s) present, skipping this cell type for this cohort")
        continue
    marker_sets[cell_type] = present

print()
print(f"Scoring {len(marker_sets)} / {len(marker_sets_raw)} cell types: {list(marker_sets)}")

## Score Cells Against Each Marker Set

`sc.tl.score_genes` scores each cell as the average expression of the marker
set minus the average expression of a random control gene set matched for
expression level.

In [ ]:
score_cols = []
for cell_type, genes in marker_sets.items():
    score_name = f"{cell_type}_score"
    sc.tl.score_genes(adata, genes, score_name=score_name, random_state=0)
    score_cols.append(score_name)

adata.obs[score_cols].describe()

## Assign Cell Type per Leiden Cluster

Scores are aggregated to the cluster level (mean per cluster) rather than
per cell. Each cluster is assigned to its highest-scoring cell type, unless
the top two scores are within `MIN_MARGIN` of each other, or the top score
itself is at or below 0 — both are labeled `Ambiguous`.

In [ ]:
MIN_MARGIN = 0.05

cluster_scores = adata.obs.groupby("leiden", observed=True)[score_cols].mean()
cluster_scores.columns = [c.replace("_score", "") for c in cluster_scores.columns]

top1_type = cluster_scores.idxmax(axis=1)
top1_score = cluster_scores.max(axis=1)
top2_score = cluster_scores.apply(lambda row: row.sort_values(ascending=False).iloc[1], axis=1)
margin = top1_score - top2_score

is_ambiguous = (margin < MIN_MARGIN) | (top1_score <= 0)

cluster_annotation = pd.DataFrame({
    "leiden": cluster_scores.index,
    "n_cells": adata.obs["leiden"].value_counts().reindex(cluster_scores.index),
    "assigned_type": np.where(is_ambiguous, "Ambiguous", top1_type),
    "top_type": top1_type,
    "top_score": top1_score.round(3),
    "second_score": top2_score.round(3),
    "margin": margin.round(3),
}).reset_index(drop=True)

cluster_annotation

## Cross-Check Cluster Scores Against Unbiased Marker Genes

Averaging continuous `score_genes` values across marker sets of different
sizes is not perfectly scale-comparable: a broad, non-specific marker set can
outscore a small but highly specific one even in a cluster the specific set
actually defines. For each cluster, this counts how many of each marker
set's genes appear in that cluster's own top-15 unbiased DE genes, and
overrides the raw-score assignment only when it is directly contradicted —
the assigned type has **zero** DE-gene hits while some other single type has
at least `MIN_DE_OVERLAP` hits.

In [ ]:
MIN_DE_OVERLAP = 2
TOP_N_DE_GENES = 15

top_de_genes = pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(TOP_N_DE_GENES)
top_de_genes.columns = top_de_genes.columns.astype(str)

de_overlap_counts = pd.DataFrame(
    {
        cell_type: [len(set(top_de_genes[cl]) & set(genes)) for cl in cluster_annotation["leiden"]]
        for cell_type, genes in marker_sets.items()
    },
    index=cluster_annotation["leiden"],
)

overrides = []
final_assigned = cluster_annotation.set_index("leiden")["assigned_type"].copy()

for cl, current in final_assigned.items():
    counts = de_overlap_counts.loc[cl]
    current_hits = counts.get(current, 0) if current != "Ambiguous" else 0

    if current_hits > 0:
        continue  # raw-score call already has direct DE support; leave it

    contradicting = counts[counts >= MIN_DE_OVERLAP]
    if len(contradicting) == 1:
        new_type = contradicting.index[0]
        overrides.append({
            "leiden": cl, "from": current, "to": new_type,
            "de_hits": ", ".join(sorted(set(top_de_genes[cl]) & set(marker_sets[new_type]))),
        })
        final_assigned[cl] = new_type

cluster_annotation["assigned_type"] = cluster_annotation["leiden"].map(final_assigned)

print(f"{len(overrides)} cluster(s) overridden by unbiased DE evidence:")
pd.DataFrame(overrides)

## Cross-Check Against the Manual `cell_type_candidate` Calls

Agreement is a sanity check on the systematic method; disagreement is not
automatically a problem, but should be inspected before trusting either
label.

In [ ]:
cluster_to_type = cluster_annotation.set_index("leiden")["assigned_type"]
adata.obs["cell_type"] = adata.obs["leiden"].map(cluster_to_type).astype("category")

pd.crosstab(adata.obs["cell_type"], adata.obs["cell_type_candidate"])

## Visualize

In [ ]:
sc.pl.umap(adata, color="cell_type", legend_loc="on data", legend_fontsize=7)
sc.pl.dotplot(adata, marker_sets, groupby="cell_type", standard_scale="var")

## Save Annotated AnnData

In [ ]:
adata.write(OUTPUT_H5AD)

summary_path = RESULTS_DIR / "cell_annotation" / f"{COHORT}_cluster_annotation_summary.csv"
cluster_annotation.to_csv(summary_path, index=False)

print(f"Saved annotated AnnData: {OUTPUT_H5AD}")
print(f"Saved cluster annotation summary -> {summary_path}")
print()
print("Cell type counts:")
print(adata.obs["cell_type"].value_counts())